In [7]:
import pandas as pd
import numpy as np
import krippendorff
from nltk.metrics.distance import masi_distance
from scipy.spatial.distance import jaccard

df = pd.read_csv('../data/AB_comparison.csv')

TECHNIQUES = [
    'Autorytet', 'Dać, aby wziąć', 'Drzwi zatrzaśnięte przed nosem',
    'Efekt „wszyscy to wiedzą”', 'Etykietowanie', 'Humor',
    'Huśtawka emocjonalna', 'Jesteśmy wyjątkowi', 'Komplementowanie',
    'Odwoływanie się do lęku', 'Odwoływanie się do uczucia miłości',
    'Okazywanie wdzięczności', 'Poczucie winy', 'Podobieństwa',
    'Przewidywanie żalu', 'Reguła lubienia', 'Reguła „my”',
    'Rozczarowanie', 'Szukamy takich jak ty', 'U nas tak się robi'
]

def parse(s):
    return frozenset(t.strip() for t in str(s).split('\n') if t.strip()) if pd.notna(s) else frozenset()

def to_vec(s):
    return np.array([1 if t in s else 0 for t in TECHNIQUES])

def jaccard_dist(a, b):
    if len(a) == 0 and len(b) == 0:
        return 0.0
    va, vb = to_vec(a), to_vec(b)
    j = jaccard(va, vb)
    return j

def masi_dist(a, b):
    if len(a) == 0 and len(b) == 0:
        return 0.0
    if len(a) == 0 or len(b) == 0:
        return 1 / len(TECHNIQUES)
    return masi_distance(a, b)

def calc_alpha(matrix, dist_func):
    """Krippendorff alpha with custom distance for set-valued data."""
    all_vals = list(set(v for row in matrix for v in row if v is not None))
    if len(all_vals) < 2:
        return np.nan
    val_to_idx = {v: i for i, v in enumerate(all_vals)}
    
    n = len(all_vals)
    precomputed = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            precomputed[i, j] = dist_func(all_vals[i], all_vals[j])
    
    # Encode as indices
    encoded = np.full((len(matrix), len(matrix[0])), np.nan)
    for i, row in enumerate(matrix):
        for j, v in enumerate(row):
            if v is not None:
                encoded[i, j] = val_to_idx[v]
    
    # Metric with krippendorff's expected signature
    def metric(v1, v2, i1=None, i2=None, n_v=None, dtype=None, **kwargs):
        return precomputed[i1, i2].astype(dtype)
    
    return krippendorff.alpha(reliability_data=encoded, level_of_measurement=metric)

def inter_alpha(df_sub, col, dist_func):
    texts = sorted(df_sub['id'].unique())
    annotators = sorted(df_sub['annotator'].unique())
    matrix = []
    for a in annotators:
        row = []
        for t in texts:
            sub = df_sub[(df_sub['annotator'] == a) & (df_sub['id'] == t)]
            row.append(parse(sub[col].iloc[0]) if len(sub) > 0 else None)
        matrix.append(row)
    return calc_alpha(matrix, dist_func)

def intra_alpha(df_sub, dist_func):
    results = {}
    for ann in df_sub['annotator'].unique():
        d = df_sub[df_sub['annotator'] == ann]
        matrix = [
            [parse(r['first_question_2_techniques']) for _, r in d.iterrows()],
            [parse(r['repeated_question_2_techniques']) for _, r in d.iterrows()]
        ]
        results[ann] = calc_alpha(matrix, dist_func)
    return results

# Groups
groups = {
    'All': df,
    'Psych+Comm': df[df['annotator_group'].isin(['psycholog', 'komunikacja'])],
    'Other3': df[df['annotator_group'].isin(['nauczyciel', 'nastolatek', 'rodzic'])]
}

# Results
rows = []
for metric_name, dist_func in [('Jaccard', jaccard_dist), ('MASI', masi_dist)]:
    intra_all = intra_alpha(df, dist_func)
    for gname, gsub in groups.items():
        intra_vals = [v for a, v in intra_all.items() if a in gsub['annotator'].values and not np.isnan(v)]
        rows.append({
            'Distance': metric_name,
            'Group': gname,
            'Inter-First': inter_alpha(gsub, 'first_question_2_techniques', dist_func),
            'Inter-Repeated': inter_alpha(gsub, 'repeated_question_2_techniques', dist_func),
            'Intra-Mean': np.mean(intra_vals),
            'Intra-Std': np.std(intra_vals)
        })

result = pd.DataFrame(rows)
print(result.to_string(index=False, float_format='%.4f'))

Distance      Group  Inter-First  Inter-Repeated  Intra-Mean  Intra-Std
 Jaccard        All       0.3186          0.3269      0.4418     0.1207
 Jaccard Psych+Comm       0.3828          0.4053      0.5138     0.0688
 Jaccard     Other3       0.2903          0.2861      0.3938     0.1240
    MASI        All       0.2087          0.2001      0.2893     0.1053
    MASI Psych+Comm       0.2600          0.2494      0.3112     0.0926
    MASI     Other3       0.1844          0.1780      0.2748     0.1106
